# Experiment - adding the UNI foundation model

Introduces [UNI](https://huggingface.co/MahmoodLab/UNI), a Vision Transformer pretrained on histopathology, as a second opinion alongside the ResNet18 tile classifier. The backbone is frozen and only a small MLP head is trained on its embeddings.

**Test F1: 0.4020**, from 0.3859 — the largest single gain of the challenge, and the point at which domain-specific pretraining clearly outperformed ImageNet features. The mixing weight is still hand-picked here.

---


## Setup and paths

Originally executed on Google Colab with the dataset on Google Drive. The mount
has been replaced by a portable path configuration: point `WSI_DATA_DIR` at a
directory holding the competition data (see `data/README.md`).


In [ ]:
# --- Paths ---------------------------------------------------------------
# Originally executed on Google Colab with the dataset on Google Drive.
# DATA_DIR must contain train_data/, test_data/ and train_labels.csv
# (see data/README.md).
import os

DATA_DIR = os.environ.get("WSI_DATA_DIR", "data")
os.makedirs("models", exist_ok=True)
os.makedirs("artifacts", exist_ok=True)


## ResNet18 Model


## Imports and Seed setting


In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
import pandas as pd

In [ ]:
SEED = 42
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


## Preprocessed Data Loading


In [ ]:
def load_npz(path):
    d = np.load(path, allow_pickle=True)
    return d["tiles"], d["labels"], d["slide_ids"]

train_tiles, train_labels, train_slide_ids = load_npz("train_tiles_raw.npz")
test_tiles, _, test_slide_ids = load_npz("test_tiles_raw_3.npz")

print(train_tiles.shape, train_labels.shape)
print(test_tiles.shape)

(3392, 256, 256, 3) (3392,)
(2547, 256, 256, 3)


## Dataset Creation


In [ ]:
class TileDataset(Dataset):
    def __init__(self, tiles, labels, transform=None):
        self.tiles = tiles
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.tiles)

    def __getitem__(self, i):
        img = torch.from_numpy(self.tiles[i]).permute(2,0,1).float() / 255.0
        if self.transform:
            img = self.transform(img)
        y = int(self.labels[i])
        return img, y

## Data Normalization and Augmentation


In [ ]:
from torchvision import transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(90),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_tf = transforms.Compose([
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

## Data loaders and model definition


In [ ]:
train_ds = TileDataset(train_tiles, train_labels, transform=train_tf)
train_loader = DataLoader(
    train_ds,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

In [ ]:
model = timm.create_model(
    "resnet18",
    pretrained=True,
    num_classes=4
).to(device)

## Class Weighting, Label Smoothing and Optimizer definition


In [ ]:
class_counts = np.bincount(train_labels, minlength=4)
class_weights = (class_counts.sum() / class_counts).astype(np.float32)
class_weights = class_weights / class_weights.mean()
weights = torch.tensor(class_weights, dtype=torch.float32, device=device)

criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)

## Model Training


In [ ]:
EPOCHS = 15

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch}/{EPOCHS} - loss: {running_loss / len(train_loader):.4f}")

torch.save(model.state_dict(), "resnet18_multiclass_secondtry_1212.pth")

Epoch 1/15 - loss: 1.4136
Epoch 2/15 - loss: 1.3824
Epoch 3/15 - loss: 1.3509
Epoch 4/15 - loss: 1.3143
Epoch 5/15 - loss: 1.2917
Epoch 6/15 - loss: 1.2616
Epoch 7/15 - loss: 1.2474
Epoch 8/15 - loss: 1.2241
Epoch 9/15 - loss: 1.2022
Epoch 10/15 - loss: 1.1743
Epoch 11/15 - loss: 1.1534
Epoch 12/15 - loss: 1.1255
Epoch 13/15 - loss: 1.1046
Epoch 14/15 - loss: 1.0859
Epoch 15/15 - loss: 1.0626


## TTA predictions


In [ ]:
import torchvision.transforms.functional as TF

@torch.no_grad()
def predict_slide_tta_meanprob(model, tiles, idxs, transform):
    imgs = torch.from_numpy(tiles[idxs]).permute(0,3,1,2).float() / 255.0
    imgs = torch.stack([transform(im) for im in imgs], dim=0)

    model.eval()
    bs = 64
    all_probs = []

    for i in range(0, imgs.size(0), bs):
        batch = imgs[i:i+bs].to(device)

        p0 = torch.softmax(model(batch), dim=1)
        p1 = torch.softmax(model(TF.hflip(batch)), dim=1)
        p2 = torch.softmax(model(TF.vflip(batch)), dim=1)

        p = (p0 + p1 + p2) / 3.0
        all_probs.append(p.cpu())

    all_probs = torch.cat(all_probs, dim=0)
    return all_probs.mean(dim=0)

In [ ]:
from collections import defaultdict

def build_slide_index(slide_ids):
    d = defaultdict(list)
    for i, sid in enumerate(slide_ids):
        d[sid].append(i)
    return d

train_slide_map = build_slide_index(train_slide_ids)
test_slide_map  = build_slide_index(test_slide_ids)

In [ ]:
model.load_state_dict(torch.load("resnet18_multiclass_secondtry_1212.pth", map_location=device))

test_preds = []
for sid, idxs in test_slide_map.items():
    prob = predict_slide_tta_meanprob(model, test_tiles, idxs, val_tf)
    test_preds.append((sid, prob.argmax().item()))

In [ ]:
label_map = {
    0: "Luminal A",
    1: "Luminal B",
    2: "HER2(+)",
    3: "Triple negative"
}

submission = pd.DataFrame({
    "sample_index": [sid for sid, _ in test_preds],
    "label": [label_map[p] for _, p in test_preds]
})

submission.to_csv("submission_resnet18_multiclass_1212_secondtry.csv", index=False)
submission.head()

,sample_index,label
0,img_0000.png,Luminal A
1,img_0001.png,Luminal B
2,img_0002.png,Luminal B
3,img_0003.png,Luminal B
4,img_0004.png,Luminal A


### Test set score: **0.3859**

## UNI Model (ViT)


In [ ]:
import torch
import numpy as np
from torchvision import models, transforms
from collections import defaultdict

device = "cuda"

# --- load model ---
model = models.resnet18(weights=None)
model.fc = torch.nn.Linear(model.fc.in_features, 4)
model.load_state_dict(torch.load("resnet18_multiclass_secondtry_1212.pth", map_location=device))
model.to(device)
model.eval()

# --- transform (IDENTICA al training) ---
transform = transforms.Compose([
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# --- load test tiles ---
tiles, _, slide_ids = load_npz("test_tiles_raw_3.npz")

slide_probs_resnet = defaultdict(list)

BS = 64
with torch.no_grad():
    for i in range(0, len(tiles), BS):
        batch = tiles[i:i+BS]
        ids   = slide_ids[i:i+BS]

        x = torch.from_numpy(batch).permute(0,3,1,2).float()/255.
        x = torch.stack([transform(im) for im in x]).to(device)

        probs = torch.softmax(model(x), dim=1).cpu().numpy()

        for sid, p in zip(ids, probs):
            slide_probs_resnet[sid].append(p)

# aggregate slide-level
slide_probs_resnet = {
    sid: np.mean(ps, axis=0)
    for sid, ps in slide_probs_resnet.items()
}

print("ResNet slides:", len(slide_probs_resnet))

ResNet slides: 477


In [ ]:
import timm
import torch

device = "cuda"

# UNI backbone (feature extractor)
uni_backbone = timm.create_model(
    "hf-hub:MahmoodLab/uni",
    pretrained=True,
    init_values=1e-5,
    dynamic_img_size=True,
    num_classes=0
)

for p in uni_backbone.parameters():
    p.requires_grad = False

uni_backbone.to(device)
uni_backbone.eval()

print("UNI backbone loaded. Output dim:", uni_backbone.num_features)

UNI backbone loaded. Output dim: 1024


In [ ]:
import torch.nn as nn

uni_classifier = nn.Sequential(
    nn.Linear(1024, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, 4)
).to(device)

uni_classifier.load_state_dict(
    torch.load("best_uni_linear_rawtiles.pth", map_location=device)
)

uni_classifier.eval()
print("UNI classifier loaded correctly (sequential)")

UNI classifier loaded correctly (sequential)


In [ ]:
from torchvision import transforms

uni_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

In [ ]:
import numpy as np
from collections import defaultdict
from PIL import Image
import torch

uni_backbone.eval()
uni_classifier.eval()

slide_probs_uni = defaultdict(list)

BS = 64
with torch.no_grad():
    for i in range(0, len(test_tiles), BS):
        batch_tiles = test_tiles[i:i+BS]
        batch_ids   = test_slide_ids[i:i+BS]

        imgs = []
        for tile in batch_tiles:
            img = Image.fromarray(tile)
            imgs.append(uni_transform(img))

        imgs = torch.stack(imgs).to(device)

        feats = uni_backbone(imgs)              # (bs, 1024)
        logits = uni_classifier(feats)          # (bs, 4)
        probs = torch.softmax(logits, dim=1).cpu().numpy()

        for sid, p in zip(batch_ids, probs):
            slide_probs_uni[sid].append(p)

# mean pooling slide-level
slide_probs_uni = {
    sid: np.mean(ps, axis=0)
    for sid, ps in slide_probs_uni.items()
}

print("UNI slides:", len(slide_probs_uni))

UNI slides: 477


In [ ]:
import pandas as pd

IDX_TO_LABEL = {
    0: "Luminal A",
    1: "Luminal B",
    2: "HER2(+)",
    3: "Triple negative"
}

ALPHA = 0.6   # ResNet weight (anything in 0.5-0.7 is reasonable)

rows = []

for sid in slide_probs_resnet:
    p_r = slide_probs_resnet[sid]
    p_u = slide_probs_uni[sid]

    p = ALPHA * p_r + (1 - ALPHA) * p_u
    pred = p.argmax()

    rows.append({
        "sample_index": sid,
        "label": IDX_TO_LABEL[pred]
    })

df = pd.DataFrame(rows).sort_values("sample_index")
df.to_csv("submission_ensemble_resnet_uni.csv", index=False)

print("Saved submission_ensemble_resnet_uni.csv")

Saved submission_ensemble_resnet_uni.csv


## Imports and Seed setting


In [ ]:
import os
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


## Preprocessed Data Loading


In [ ]:
def load_npz(path):
    d = np.load(path, allow_pickle=True)
    return d["tiles"], d["labels"], d["slide_ids"]

train_tiles, train_labels, train_slide_ids = load_npz("train_tiles_raw.npz")
test_tiles, _, test_slide_ids = load_npz("test_tiles_raw_3.npz")

print("Train tiles:", train_tiles.shape)
print("Train labels:", train_labels.shape)
print("Test tiles:", test_tiles.shape)

Train tiles: (3392, 256, 256, 3)
Train labels: (3392,)
Test tiles: (2547, 256, 256, 3)


In [ ]:
from collections import defaultdict

tiles_per_slide = defaultdict(list)

for tile, lab, sid in zip(train_tiles, train_labels, train_slide_ids):
    tiles_per_slide[sid].append((tile, lab))

all_slides = sorted(tiles_per_slide.keys())
print("Unique train slides:", len(all_slides))

Unique train slides: 627


## Train and Validation Split (slide-level)


In [ ]:
from sklearn.model_selection import train_test_split

train_slides, val_slides = train_test_split(
    all_slides,
    test_size=0.20,
    random_state=SEED,
    shuffle=True
)

print("Train slides:", len(train_slides))
print("Val slides:", len(val_slides))

Train slides: 501
Val slides: 126


## HuggingFace Login and Model Creation


In [ ]:
from huggingface_hub import login

login()

In [ ]:
import timm
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform

model = timm.create_model(
    "hf-hub:MahmoodLab/uni",
    pretrained=True,
    init_values=1e-5,
    dynamic_img_size=True,
    num_classes=0
)

model.eval()
model.to(device)

for p in model.parameters():
    p.requires_grad = False

transform = create_transform(
    **resolve_data_config(model.pretrained_cfg, model=model)
)

embed_dim = model.num_features
print("UNI embedding dim:", embed_dim)

config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.21G [00:00<?, ?B/s]

UNI embedding dim: 1024


In [ ]:
from PIL import Image

@torch.no_grad()
def extract_features(slide_ids_subset):
    feats = {}
    labels_out = {}

    for sid in slide_ids_subset:
        tiles = tiles_per_slide[sid]
        imgs = []

        for t, _ in tiles:
            # t: numpy array HWC uint8
            img = Image.fromarray(t)
            img = transform(img)         # torch.Tensor [3, 224, 224]
            imgs.append(img)

        imgs = torch.stack(imgs).to(device)   # (N_tiles, 3, 224, 224)
        emb = model(imgs)                      # (N_tiles, 1024)

        feats[sid] = emb.cpu()
        labels_out[sid] = tiles[0][1]

    return feats, labels_out

In [ ]:
train_feats, train_labels_slide = extract_features(train_slides)
val_feats, val_labels_slide = extract_features(val_slides)

print("Train slides features:", len(train_feats))
print("Val slides features:", len(val_feats))

Train slides features: 501
Val slides features: 126


## Dataset creation


In [ ]:
from torch.utils.data import Dataset
import torch

class SlideDataset(Dataset):
    def __init__(self, feats, labels):
        self.ids = list(feats.keys())
        self.feats = feats
        self.labels = labels

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        sid = self.ids[idx]
        x = self.feats[sid].mean(dim=0)           # (1024,)
        y = torch.tensor(self.labels[sid]).long()
        return x, y

## Data Loader creation


In [ ]:
from torch.utils.data import DataLoader

train_ds = SlideDataset(train_feats, train_labels_slide)
val_ds   = SlideDataset(val_feats, val_labels_slide)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=64, shuffle=False)

## Classifier definition


In [ ]:
import torch.nn as nn

classifier = nn.Sequential(
    nn.Linear(embed_dim, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, 4)
).to(device)

## Class Weights and Optimizer setting


In [ ]:
from collections import Counter

counts = Counter(train_labels_slide.values())
weights = torch.tensor(
    [1.0 / counts[i] for i in range(4)],
    dtype=torch.float32,
    device=device
)
weights = weights / weights.sum()

criterion = nn.CrossEntropyLoss(weight=weights)

optimizer = torch.optim.AdamW(
    classifier.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

print("Class weights:", weights)

Class weights: tensor([0.1905, 0.1516, 0.2134, 0.4445], device='cuda:0')


In [ ]:
from sklearn.metrics import f1_score

def evaluate():
    classifier.eval()
    preds, targets = [], []

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            y = y.to(device)

            logits = classifier(x)
            preds.extend(logits.argmax(dim=1).cpu().numpy())
            targets.extend(y.cpu().numpy())

    return f1_score(targets, preds, average="macro")

## Model Training


In [ ]:
EPOCHS = 25
best_f1 = -1.0

for ep in range(1, EPOCHS + 1):
    classifier.train()
    loss_sum = 0.0

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        out = classifier(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item()

    val_f1 = evaluate()

    print(f"[Ep {ep:02d}] Train loss {loss_sum/len(train_loader):.4f} | Val F1 {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(classifier.state_dict(), "best_uni_linear_rawtiles.pth")

[Ep 01] Train loss 1.4990 | Val F1 0.3261
[Ep 02] Train loss 1.2913 | Val F1 0.3079
[Ep 03] Train loss 1.2169 | Val F1 0.3808
[Ep 04] Train loss 1.1433 | Val F1 0.4067
[Ep 05] Train loss 1.1589 | Val F1 0.3825
[Ep 06] Train loss 1.0680 | Val F1 0.4787
[Ep 07] Train loss 1.0039 | Val F1 0.4643
[Ep 08] Train loss 0.9763 | Val F1 0.4391
[Ep 09] Train loss 0.9575 | Val F1 0.5275
[Ep 10] Train loss 0.9101 | Val F1 0.4459
[Ep 11] Train loss 0.8810 | Val F1 0.4660
[Ep 12] Train loss 0.8120 | Val F1 0.4801
[Ep 13] Train loss 0.8463 | Val F1 0.4331
[Ep 14] Train loss 0.8323 | Val F1 0.4275
[Ep 15] Train loss 0.8137 | Val F1 0.5149
[Ep 16] Train loss 0.7199 | Val F1 0.5088
[Ep 17] Train loss 0.7231 | Val F1 0.5088
[Ep 18] Train loss 0.6997 | Val F1 0.4948
[Ep 19] Train loss 0.6689 | Val F1 0.5335
[Ep 20] Train loss 0.6307 | Val F1 0.4925
[Ep 21] Train loss 0.5846 | Val F1 0.5628
[Ep 22] Train loss 0.5894 | Val F1 0.5330
[Ep 23] Train loss 0.5639 | Val F1 0.4911
[Ep 24] Train loss 0.4908 | Val F1

In [ ]:
classifier.load_state_dict(
    torch.load("best_uni_linear_rawtiles.pth", map_location=device)
)
classifier.eval()

Sequential(
  (0): Linear(in_features=1024, out_features=256, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.4, inplace=False)
  (3): Linear(in_features=256, out_features=4, bias=True)
)

In [ ]:
from collections import defaultdict

test_tiles_per_slide = defaultdict(list)

for tile, sid in zip(test_tiles, test_slide_ids):
    test_tiles_per_slide[sid].append(tile)

test_slides = sorted(test_tiles_per_slide.keys())
print("Test slides:", len(test_slides))

Test slides: 477


In [ ]:
@torch.no_grad()
def predict_test_slides():
    preds = {}

    for sid in test_slides:
        tiles = test_tiles_per_slide[sid]
        imgs = []

        for t in tiles:
            img = Image.fromarray(t)
            img = transform(img)
            imgs.append(img)

        imgs = torch.stack(imgs).to(device)     # (N_tiles, 3, 224, 224)

        emb = model(imgs)                       # (N_tiles, 1024)
        slide_emb = emb.mean(dim=0, keepdim=True)  # (1, 1024)

        logits = classifier(slide_emb)
        pred = logits.argmax(dim=1).item()

        preds[sid] = pred

    return preds

In [ ]:
test_preds = predict_test_slides()
print("Inference done on test set")

Inference done on test set


In [ ]:
import pandas as pd

label_map = {
    0: "Luminal A",
    1: "Luminal B",
    2: "HER2(+)",
    3: "Triple negative"
}

submission = pd.DataFrame({
    "sample_index": list(test_preds.keys()),
    "label": [label_map[v] for v in test_preds.values()]
})

submission = submission.sort_values("sample_index").reset_index(drop=True)
submission.to_csv("submission_uni_mean.csv", index=False)

submission.head()

,sample_index,label
0,img_0000.png,Luminal B
1,img_0001.png,Luminal A
2,img_0002.png,Triple negative
3,img_0003.png,Luminal B
4,img_0004.png,Luminal A


### Test set score: **0.3777**

## ResNet + UNI Ensemble


In [ ]:
import torch
import numpy as np
from torchvision import models, transforms
from collections import defaultdict

device = "cuda"

# --- load model ---
model = models.resnet18(weights=None)
model.fc = torch.nn.Linear(model.fc.in_features, 4)
model.load_state_dict(torch.load("resnet18_multiclass_secondtry_1212.pth", map_location=device))
model.to(device)
model.eval()

# --- transform (IDENTICA al training) ---
transform = transforms.Compose([
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# --- load test tiles ---
tiles, _, slide_ids = load_npz("test_tiles_raw_3.npz")

slide_probs_resnet = defaultdict(list)

BS = 64
with torch.no_grad():
    for i in range(0, len(tiles), BS):
        batch = tiles[i:i+BS]
        ids   = slide_ids[i:i+BS]

        x = torch.from_numpy(batch).permute(0,3,1,2).float()/255.
        x = torch.stack([transform(im) for im in x]).to(device)

        probs = torch.softmax(model(x), dim=1).cpu().numpy()

        for sid, p in zip(ids, probs):
            slide_probs_resnet[sid].append(p)

# aggregate slide-level
slide_probs_resnet = {
    sid: np.mean(ps, axis=0)
    for sid, ps in slide_probs_resnet.items()
}

print("ResNet slides:", len(slide_probs_resnet))

ResNet slides: 477


In [ ]:
import timm
import torch

device = "cuda"

# UNI backbone (feature extractor)
uni_backbone = timm.create_model(
    "hf-hub:MahmoodLab/uni",
    pretrained=True,
    init_values=1e-5,
    dynamic_img_size=True,
    num_classes=0
)

for p in uni_backbone.parameters():
    p.requires_grad = False

uni_backbone.to(device)
uni_backbone.eval()

print("UNI backbone loaded. Output dim:", uni_backbone.num_features)

UNI backbone loaded. Output dim: 1024


In [ ]:
import torch.nn as nn

uni_classifier = nn.Sequential(
    nn.Linear(1024, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, 4)
).to(device)

uni_classifier.load_state_dict(
    torch.load("best_uni_linear_rawtiles.pth", map_location=device)
)

uni_classifier.eval()
print("UNI classifier loaded correctly (sequential)")

UNI classifier loaded correctly (sequential)


In [ ]:
from torchvision import transforms

uni_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

In [ ]:
import numpy as np
from collections import defaultdict
from PIL import Image
import torch

uni_backbone.eval()
uni_classifier.eval()

slide_probs_uni = defaultdict(list)

BS = 64
with torch.no_grad():
    for i in range(0, len(test_tiles), BS):
        batch_tiles = test_tiles[i:i+BS]
        batch_ids   = test_slide_ids[i:i+BS]

        imgs = []
        for tile in batch_tiles:
            img = Image.fromarray(tile)
            imgs.append(uni_transform(img))

        imgs = torch.stack(imgs).to(device)

        feats = uni_backbone(imgs)              # (bs, 1024)
        logits = uni_classifier(feats)          # (bs, 4)
        probs = torch.softmax(logits, dim=1).cpu().numpy()

        for sid, p in zip(batch_ids, probs):
            slide_probs_uni[sid].append(p)

# mean pooling slide-level
slide_probs_uni = {
    sid: np.mean(ps, axis=0)
    for sid, ps in slide_probs_uni.items()
}

print("UNI slides:", len(slide_probs_uni))

UNI slides: 477


In [ ]:
import pandas as pd

IDX_TO_LABEL = {
    0: "Luminal A",
    1: "Luminal B",
    2: "HER2(+)",
    3: "Triple negative"
}

ALPHA = 0.6   # ResNet weight (anything in 0.5-0.7 is reasonable)

rows = []

for sid in slide_probs_resnet:
    p_r = slide_probs_resnet[sid]
    p_u = slide_probs_uni[sid]

    p = ALPHA * p_r + (1 - ALPHA) * p_u
    pred = p.argmax()

    rows.append({
        "sample_index": sid,
        "label": IDX_TO_LABEL[pred]
    })

df = pd.DataFrame(rows).sort_values("sample_index")
df.to_csv("submission_ensemble_resnet_uni.csv", index=False)

print("Saved submission_ensemble_resnet_uni.csv")

Saved submission_ensemble_resnet_uni.csv
